In [ ]:
import pandas as pd
import numpy as np
import os
import keras
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

#%%
base_dir = r"C:\Users\asus\Desktop\cifar10-raw-images - Copy\images"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
#%%
def load_data_from_folder(dataset_path):
    images_list = []
    labels_list = []
    classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    classes.sort()
    class_idx = {name: idx for idx, name in enumerate(classes)}

    for class_name in classes:
        class_path = os.path.join(dataset_path, class_name)
        if not os.path.isdir(class_path):
            continue
        label = class_idx[class_name]

        for file_name in os.listdir(class_path):
            if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
                file_path = os.path.join(class_path, file_name)
                try:
                    with Image.open(file_path) as img:
                        img = img.convert('RGB')
                        if img.size != (32, 32):
                            img = img.resize((32, 32))

                        img_array = np.array(img)
                        images_list.append(img_array)
                        labels_list.append(label)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    continue

    return np.array(images_list), np.array(labels_list)


print("@@@@......dar hal khandan file train")
train_x, train_y = load_data_from_folder(train_dir)


#%%
print("@@@@......dar hal khandan file test")
test_x, test_y = load_data_from_folder(test_dir)
#%%
print("Train shape:", train_x.shape)

print("Test shape:", test_x.shape)

#%%
train_x = train_x / 255.0
test_x = test_x / 255.0

#%%
model = keras.models.Sequential([
    keras.layers.Input(shape=train_x.shape[1:]),
    keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='SAME'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='SAME'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.2),

    keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='SAME'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='SAME'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.3),

    keras.layers.Conv2D(128, kernel_size=(3, 3), activation='relu', padding='SAME'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, kernel_size=(3, 3), activation='relu', padding='SAME'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.3),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')
])
#%%
model.compile(loss=keras.losses.SparseCategoricalCrossentropy(),
              optimizer=keras.optimizers.Adam(),
              metrics=['accuracy'])
#%%
call_back_list = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(monitor="val_loss", filepath="best_model.keras", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-5)
]
#%%
history = model.fit(train_x, train_y, batch_size=64, epochs=15,
                    validation_data=(test_x, test_y), callbacks=call_back_list)

test_loss, test_acc = model.evaluate(test_x, test_y)
print("test_loss:", test_loss)
print("test_acc:", test_acc)
#%%
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label="train")
plt.plot(history.history['val_accuracy'], label="test")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label="train")
plt.plot(history.history['val_loss'], label="test")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

classes = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
classes.sort()
print("Classes:", classes)

#%%
class_names=[ 'Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']                                  
def guess_image(models, path_image, class_names):
    img = Image.open(path_image)
    img = img.convert('RGB')
    img_res = img.resize((32, 32))
    array_image = np.array(img_res, dtype=np.float32) / 255.0
    batch_image = np.expand_dims(array_image, axis=0)

    predictions = model.predict(batch_image)
    prediction_idx_class = np.argmax(predictions[0])
    confidence = predictions[0][prediction_idx_class] * 100
    class_name_ma = class_names[prediction_idx_class]

    print("natije pishbini:", class_name_ma)
    print("darsad etminan: {:.2f}%".format(confidence))
    return class_name_ma, confidence
guess_image(model, r"C:\Users\asus\Pictures\photo.jpg",class_names)


#%%

